[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter3/parse-tables.ipynb)

# Chapter 3: Table Extraction from Documents

Tables are one of the hardest document elements to handle in RAG pipelines. Standard text extraction treats table content as unstructured text, losing the row/column relationships that give the data meaning. Proper table extraction preserves this structure, enabling accurate retrieval and question-answering over tabular data.

This notebook demonstrates how to extract tables from PDF documents using Docling for structured data retrieval in RAG systems.

**What you'll learn:**
- Use Docling's document converter to parse PDFs with table detection
- Export extracted tables to pandas DataFrames
- Handle multi-page documents with multiple tables

**Prerequisites:** `pip install docling`

## Setup

We install Docling, a document-understanding library that uses deep learning to detect and extract tables from PDFs with high fidelity.

In [1]:
!pip install --quiet docling


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import requests

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

## Parse Tables from PDF

We download a full textbook PDF and run it through Docling's converter, which identifies page structure and extracts every table it finds.

In [3]:
url = "https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf"
local_file = "sutter_barto.pdf"
with requests.get(url, stream=True) as response:
    response.raise_for_status()
    with open(local_file, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

# Create docling PDF Pipeline
pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True

# Convert document
res = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
).convert(local_file)
doc = res.document

/Users/ofer/miniconda3/envs/py312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/ofer/miniconda3/envs/py312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/ofer/miniconda3/envs/py312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/ofer/miniconda3/envs/py312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/ofer/mini

## Display Extracted Tables

Once tables are extracted, Docling can export them directly to pandas DataFrames for inspection and downstream processing.

In [4]:
table = doc.tables[13]
table_df = table.export_to_dataframe()
table_df

,Program,Hidden Units,Training Games,Opponents,Results
0,TD-Gam 0.0,40,"300,000",other programs,tied for best
1,TD-Gam 1.0,80,"300,000","Robertie, Magriel, ...",- 13 pts / 51 games
2,TD-Gam 2.0,40,"800,000",various Grandmasters,- 7 pts / 38 games
3,TD-Gam 2.1,80,"1,500,000",Robertie,- 1 pt / 40 games
4,TD-Gam 3.0,80,"1,500,000",Kazaros,+6 pts / 20 games


In [5]:
table = doc.tables[10]
table_df = table.export_to_dataframe()
print(table_df.shape)

(0, 0)
